# 05 - Live Trading Bridge

Take target weights through the **execution layer**: pre-trade risk checks, weight->order translation, the order-lifecycle state machine, state persistence, and an Alpaca **paper** connection (auto-skipped offline). One full rebalance cycle.

> **Data input:** Set exactly one of `QUANTCORTEX_PRICES_CSV` or `QUANTCORTEX_LIVE_YFINANCE=1`. No market data or executed outputs are committed.


In [ ]:
import logging
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

logging.getLogger("hmmlearn").setLevel(logging.ERROR)
logging.getLogger("yfinance").setLevel(logging.CRITICAL)
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import matplotlib

try:
    shell = get_ipython()
except NameError:
    shell = None
if shell is None:
    matplotlib.use("Agg")
else:
    shell.run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt

# Put the repository root on the path when running from research/.
for candidate in ("..", "."):
    absolute = os.path.abspath(candidate)
    if absolute not in sys.path:
        sys.path.insert(0, absolute)

from quantcortex.backtest.metrics.plotting import (
    INK, NEGATIVE_RED, REFERENCE_BLUE, apply_plot_style, style_axis,
)
from quantcortex.data.local_csv import load_ohlcv_csv, load_price_matrix

apply_plot_style("notebook")

PRICE_CSV = os.environ.get("QUANTCORTEX_PRICES_CSV")
OHLCV_CSV = os.environ.get("QUANTCORTEX_OHLCV_CSV")
LIVE_YFINANCE = os.environ.get("QUANTCORTEX_LIVE_YFINANCE") == "1"

if (PRICE_CSV is not None) == LIVE_YFINANCE:
    raise RuntimeError(
        "Set exactly one of QUANTCORTEX_PRICES_CSV=/path/to/prices.csv "
        "or QUANTCORTEX_LIVE_YFINANCE=1."
    )


def load_prices(symbols, start="2018-01-01"):
    """Load adjusted closes from the explicitly selected real-data source."""
    if PRICE_CSV is not None:
        return load_price_matrix(PRICE_CSV, symbols=list(symbols), start=start)

    print(
        "Using live Yahoo Finance data through yfinance; review the provider "
        "terms and confirm that your use is permitted."
    )
    from quantcortex.data.providers.yfinance_provider import YFinanceProvider

    prices = YFinanceProvider().get_prices(list(symbols), start=start)
    if prices is None or prices.empty:
        raise RuntimeError("yfinance returned no prices")
    prices = prices.dropna(how="all").ffill(limit=5).dropna()
    if prices.shape[0] <= 120:
        raise RuntimeError("insufficient complete price history from yfinance")
    return prices


def load_ohlcv(symbol, start="2018-01-01"):
    """Load one symbol's real OHLCV data for feature engineering."""
    if OHLCV_CSV is not None:
        return load_ohlcv_csv(OHLCV_CSV, start=start)
    if not LIVE_YFINANCE:
        raise RuntimeError(
            "Set QUANTCORTEX_OHLCV_CSV=/path/to/ohlcv.csv for the Alpha158 cell."
        )

    from quantcortex.data.providers.yfinance_provider import YFinanceProvider

    frame = YFinanceProvider().fetch_ohlcv([symbol], start=start).get(symbol)
    if frame is None or frame.empty:
        raise RuntimeError(f"yfinance returned no OHLCV data for {symbol}")
    return frame.dropna()


selected_source = (
    f"local CSV {Path(PRICE_CSV).expanduser().resolve()}"
    if PRICE_CSV is not None
    else "explicit live yfinance download"
)
print(f"quantcortex research environment ready; source: {selected_source}")


In [ ]:
symbols = ["QQQ","VGT","GLD","TLT","SPY","VIG"]
prices = load_prices(symbols)
from quantcortex.strategies.multi_asset_rotation import MultiAssetRotation

weekly = prices.index[prices.index.weekday == 0]
strat = MultiAssetRotation()
W = strat.generate_weights(prices, weekly)
if W.empty:
    raise RuntimeError("strategy produced no weekly targets")
# Use the latest target even when it is flat. Reusing an older invested row
# would bypass a current regime or risk-off liquidation signal.
target = W.iloc[-1]
target = target[target.abs() > 1e-9]
print(f"latest target weights (as of {W.index[-1].date()}):")
print(target.round(3).to_string() if not target.empty else "(flat)")

## Pre-trade risk gate
The last line of defence before any order is routed.

In [ ]:
from quantcortex.execution.pre_trade_risk import PreTradeRiskCheck
from quantcortex.portfolio.base import PortfolioMode

w_vec = target.reindex(symbols).fillna(0.0).to_numpy()
checker = PreTradeRiskCheck(
    max_position_weight=strat.max_position_weight, max_gross=1.0
)
ok, violations = checker.check_weights(w_vec, mode=PortfolioMode.LONG_ONLY)
print("pre-trade weight check ok:", ok, "| violations:", violations)

## Translate target weights into orders

In [ ]:
from quantcortex.execution.position_manager import PositionManager

capital = 100_000.0
last_px = prices.iloc[-1]
pm = PositionManager()
orders = pm.target_weights_to_orders(target, last_px, capital=capital, current_positions={})
checker.assert_safe(
    weights=w_vec, mode=PortfolioMode.LONG_ONLY,
    orders=orders, prices=last_px, capital=capital, current_positions={},
)
for o in orders:
    print("  %4s %8.1f  %s" % (o["side"].value, o["quantity"], o["symbol"]))

## Order lifecycle: NEW -> SUBMITTED -> FILLED

In [ ]:
from quantcortex.execution.order_manager import OrderManager, OrderSide

om = OrderManager()
for i, o in enumerate(orders):
    oid = f"ord-{i:03d}"
    om.create_order(oid, o["symbol"], OrderSide(o["side"]), float(o["quantity"]))
    om.submit(oid)
    om.fill(oid, fill_price=float(last_px[o["symbol"]]))
states = {o.order_id: o.status.value for o in om.orders}
print("final order states:", states)
assert all(s == "FILLED" for s in states.values())
print(f"all {len(states)} generated orders FILLED.")

## Persist state across restarts (Redis, file fallback offline)

In [ ]:
from tempfile import TemporaryDirectory

from quantcortex.execution.state_persistence import StatePersistence

positions = {
    o["symbol"]: (1.0 if o["side"] is OrderSide.BUY else -1.0)
    * float(o["quantity"])
    for o in orders
}
with TemporaryDirectory(prefix="quantcortex-notebook-") as state_dir:
    state_path = Path(state_dir) / "state.json"
    sp = StatePersistence(
        url=None, namespace="notebook", file_path=state_path
    )
    sp.save_positions(positions)
    restored = sp.load_positions()
    print("persistence backend:", sp.backend)
    print("restored positions:", restored)

## Alpaca paper connection (auto-skipped without credentials)

In [ ]:
import os

from quantcortex.execution.brokers.alpaca_broker import AlpacaBroker

if os.getenv("ALPACA_API_KEY") and os.getenv("ALPACA_SECRET_KEY"):
    broker = AlpacaBroker(paper=True)
    try:
        broker.connect()
        acct = broker.get_account()
        print(f"Connected to Alpaca paper. Equity=${acct.equity:,.2f}")
    except Exception as e:
        print("Alpaca connection failed:", e)
    finally:
        broker.disconnect()
else:
    print("No ALPACA_API_KEY set -> running offline.")
    print("Set credentials in .env to route this rebalance to Alpaca paper.")
print("\none rebalance cycle complete (paper/offline).")